# Student Performance — Notebook 2: Model Training
**Project:** End-to-End AI Framework for Student Performance Analysis
**Student:** Megha Deopa | PRN: 2405020011520 | MBA AI/ML July 2024

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models
from sklearn.linear_model     import LinearRegression, Ridge, Lasso
from sklearn.neighbors        import KNeighborsRegressor
from sklearn.tree             import DecisionTreeRegressor
from sklearn.ensemble         import RandomForestRegressor, AdaBoostRegressor
from xgboost                  import XGBRegressor
from catboost                 import CatBoostRegressor

# Preprocessing
from sklearn.preprocessing    import OneHotEncoder, StandardScaler
from sklearn.compose          import ColumnTransformer
from sklearn.model_selection  import train_test_split

print('All libraries imported!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data/stud.csv')
print('Shape:', df.shape)
df.head()

## 3. Prepare X (Features) and y (Target)

In [ ]:
# X = all columns EXCEPT math_score (our target)
X = df.drop(columns=['math_score'], axis=1)

# y = math_score (what we want to predict)
y = df['math_score']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFeatures used:')
print(X.columns.tolist())

## 4. Preprocessing — OneHotEncoding + StandardScaler

In [ ]:
# Separate numerical and categorical columns
num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

print('Numerical features:', list(num_features))
print('Categorical features:', list(cat_features))

# Define transformers
numeric_transformer = StandardScaler()   # Normalises numerical columns
oh_transformer      = OneHotEncoder()    # Converts text to binary columns

# Combine both transformers
preprocessor = ColumnTransformer([
    ('OneHotEncoder',  oh_transformer,      cat_features),
    ('StandardScaler', numeric_transformer, num_features),
])

# Apply transformations
X = preprocessor.fit_transform(X)
print('\nAfter preprocessing, X shape:', X.shape)
print('(7 original columns expanded to', X.shape[1], 'after OneHotEncoding)')

## 5. Train-Test Split (80% Train, 20% Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training samples:', X_train.shape[0])
print('Testing samples: ', X_test.shape[0])

## 6. Evaluation Function

In [ ]:
def evaluate_model(true, predicted):
    """Returns MAE, RMSE, and R2 score"""
    mae  = mean_absolute_error(true, predicted)
    mse  = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2   = r2_score(true, predicted)
    return mae, rmse, r2

print('Evaluation function defined!')

## 7. Train and Compare All 9 Regression Models

In [ ]:
models = {
    'Linear Regression':      LinearRegression(),
    'Lasso':                   Lasso(),
    'Ridge':                   Ridge(),
    'K-Neighbors Regressor':   KNeighborsRegressor(),
    'Decision Tree':           DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'XGBRegressor':            XGBRegressor(),
    'CatBoosting Regressor':   CatBoostRegressor(verbose=False),
    'AdaBoost Regressor':      AdaBoostRegressor()
}

model_list  = []
r2_list     = []
results     = []

print(f"{'Model':<28} {'Train R2':>10} {'Test R2':>10} {'MAE':>8} {'RMSE':>8}")
print('=' * 70)

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)

    # Predict on train and test
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # Evaluate
    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae,  test_rmse,  test_r2  = evaluate_model(y_test,  y_test_pred)

    model_list.append(name)
    r2_list.append(test_r2)
    results.append({
        'Model':     name,
        'Train R2':  round(train_r2, 4),
        'Test R2':   round(test_r2,  4),
        'Test MAE':  round(test_mae, 4),
        'Test RMSE': round(test_rmse,4)
    })

    print(f"{name:<28} {train_r2:>10.4f} {test_r2:>10.4f} {test_mae:>8.4f} {test_rmse:>8.4f}")

## 8. Results Summary Table

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='Test R2', ascending=False)
print('Model Comparison (sorted by Test R2 — highest to lowest):')
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
plt.figure(figsize=(14, 7))
colors = ['#2ecc71' if r == max(r2_list) else '#3498db' for r in r2_list]
bars = plt.bar(model_list, r2_list, color=colors, alpha=0.85, width=0.6)

for bar, r2 in zip(bars, r2_list):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.005,
             f'{r2:.4f}', ha='center', fontsize=9)

plt.title('Test R² Score Comparison — All 9 Models', fontsize=14)
plt.ylabel('R² Score')
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

best_model = results_df.iloc[0]['Model']
best_r2    = results_df.iloc[0]['Test R2']
print(f'\n🏆 Best Model: {best_model} with Test R² = {best_r2}')

## 9. Final Model — Linear Regression

In [ ]:
# Train the best model (Linear Regression)
lin_model = LinearRegression(fit_intercept=True)
lin_model.fit(X_train, y_train)

y_pred = lin_model.predict(X_test)
score  = r2_score(y_test, y_pred) * 100

print(f'Linear Regression — Test R² = {score:.2f}%')
print(f'This means the model explains {score:.2f}% of variation in student math scores')

### 9.1 Actual vs Predicted Scatter Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Math Score', fontsize=12)
axes[0].set_ylabel('Predicted Math Score', fontsize=12)
axes[0].set_title('Actual vs Predicted — Linear Regression', fontsize=13)
axes[0].grid(alpha=0.3)

# Regression plot
sns.regplot(x=y_test, y=y_pred, ci=None, color='red', ax=axes[1])
axes[1].set_xlabel('Actual Math Score', fontsize=12)
axes[1].set_ylabel('Predicted Math Score', fontsize=12)
axes[1].set_title('Regression Line — Actual vs Predicted', fontsize=13)

plt.suptitle(f'Linear Regression Performance (R² = {score:.2f}%)', fontsize=14)
plt.tight_layout()
plt.show()

### 9.2 Prediction Comparison Table

In [ ]:
pred_df = pd.DataFrame({
    'Actual Value':    y_test.values,
    'Predicted Value': y_pred.round(2),
    'Difference':      (y_test.values - y_pred).round(2)
})

print('Sample of Actual vs Predicted (first 20 rows):')
print(pred_df.head(20).to_string(index=False))

print(f'\nMean Absolute Error: {abs(pred_df["Difference"]).mean():.2f} score points')
print('This means predictions are within ~5 score points on average')